In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\ASUS\Intelligent-Credit-Risk-System\data\credit_data.csv")

print(df.shape)
df.head()

(104805, 12)


,Id,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
1,2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
2,3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
3,4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0
4,5,0,0.213179,74,0,0.375607,3500.0,3,0,1,0,1.0


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104805 entries, 0 to 104804
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Id                                    104805 non-null  int64  
 1   SeriousDlqin2yrs                      104805 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  104805 non-null  float64
 3   age                                   104805 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  104805 non-null  int64  
 5   DebtRatio                             104805 non-null  float64
 6   MonthlyIncome                         84024 non-null   float64
 7   NumberOfOpenCreditLinesAndLoans       104805 non-null  int64  
 8   NumberOfTimes90DaysLate               104805 non-null  int64  
 9   NumberRealEstateLoansOrLines          104805 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  104805 non-null  int64  
 11  

In [51]:
df.isnull().sum()


Id                                          0
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           20781
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       2749
dtype: int64

In [52]:
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

# Verify
df.isnull().sum()

Id                                      0
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

In [53]:
df = df.drop('Id', axis=1)

print(df.shape)

(104805, 11)


In [54]:
# DebtRatio
Q1 = df['DebtRatio'].quantile(0.25)
Q3 = df['DebtRatio'].quantile(0.75)
IQR = Q3 - Q1
df['DebtRatio'] = df['DebtRatio'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# RevolvingUtilizationOfUnsecuredLines
Q1 = df['RevolvingUtilizationOfUnsecuredLines'].quantile(0.25)
Q3 = df['RevolvingUtilizationOfUnsecuredLines'].quantile(0.75)
IQR = Q3 - Q1
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# MonthlyIncome
Q1 = df['MonthlyIncome'].quantile(0.25)
Q3 = df['MonthlyIncome'].quantile(0.75)
IQR = Q3 - Q1
df['MonthlyIncome'] = df['MonthlyIncome'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

print("Outliers removed!")

Outliers removed!


In [55]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:", list(X.columns))

X shape: (104805, 10)
y shape: (104805,)

Features: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [56]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling done!")
print("X_scaled shape:", X_scaled.shape)

Scaling done!
X_scaled shape: (104805, 10)


In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (83844, 10)
X_test: (20961, 10)
y_train: (83844,)
y_test: (20961,)


In [58]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_smote.value_counts().to_dict())

Before SMOTE: {0: 78218, 1: 5626}
After SMOTE: {0: 78218, 1: 78218}


In [59]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train_smote, y_train_smote)

print("XGBoost training done!")

XGBoost training done!


In [60]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_xgb = xgb.predict(X_test)

print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

[[19121   516]
 [  894   430]]
              precision    recall  f1-score   support

           0       0.96      0.97      0.96     19637
           1       0.45      0.32      0.38      1324

    accuracy                           0.93     20961
   macro avg       0.70      0.65      0.67     20961
weighted avg       0.92      0.93      0.93     20961



In [61]:
y_prob = xgb.predict_proba(X_test)[:, 1]
risk_score = y_prob * 100

risk_category = np.where(risk_score < 30, "Low Risk",
                np.where(risk_score < 60, "Medium Risk", "High Risk"))

results = pd.DataFrame({
    "Probability": y_prob,
    "Risk Score": risk_score,
    "Risk Category": risk_category
})

print(results.head(10))
print("\nCategory Distribution:")
print(results['Risk Category'].value_counts())

   Probability  Risk Score Risk Category
0     0.025967    2.596685      Low Risk
1     0.219150   21.915037      Low Risk
2     0.011468    1.146773      Low Risk
3     0.118465   11.846451      Low Risk
4     0.274099   27.409912      Low Risk
5     0.242019   24.201893      Low Risk
6     0.055856    5.585635      Low Risk
7     0.142070   14.206986      Low Risk
8     0.024968    2.496778      Low Risk
9     0.096078    9.607842      Low Risk

Category Distribution:
Risk Category
Low Risk       18435
Medium Risk     1955
High Risk        571
Name: count, dtype: int64


In [62]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

df['Cluster'] = clusters

print("Cluster Distribution:")
print(df['Cluster'].value_counts())
print("\nCluster Means:")
print(df.groupby('Cluster').mean().round(2))

Cluster Distribution:
Cluster
2    41303
1    32259
0    31243
Name: count, dtype: int64

Cluster Means:
         SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines    age  \
Cluster                                                                  
0                    0.14                                  0.63  40.11   
1                    0.05                                  0.26  51.51   
2                    0.02                                  0.14  62.27   

         NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
Cluster                                                                   
0                                        0.94       0.51        4192.42   
1                                        0.27       0.42        8978.00   
2                                        0.13       0.97        4725.43   

         NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
Cluster                                                             
0         

In [63]:
import joblib

joblib.dump(xgb, r"C:\Users\ASUS\Intelligent-Credit-Risk-System\models\model.pkl")
joblib.dump(scaler, r"C:\Users\ASUS\Intelligent-Credit-Risk-System\models\scaler.pkl")

print("model.pkl saved!")
print("scaler.pkl saved!")

model.pkl saved!
scaler.pkl saved!


In [64]:
# Text generation LSTM MODEL
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Load training data from file
with open(r"C:\Users\ASUS\Intelligent-Credit-Risk-System\data\training.txt", "r") as f:
    sentences = f.readlines()

sentences = [s.strip() for s in sentences if s.strip()]

print("Total sentences:", len(sentences))

# Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size:", vocab_size)

Total sentences: 114
Vocabulary Size: 488


In [65]:
# Create input sequences
input_sequences = []

for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

print("Total sequences:", len(input_sequences))

# Pad sequences
max_seq_len = max(len(x) for x in input_sequences)
print("Max sequence length:", max_seq_len)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
print("Input sequences shape:", input_sequences.shape)


Total sequences: 904
Max sequence length: 14
Input sequences shape: (904, 14)


In [66]:
import numpy as np

X_lstm = input_sequences[:, :-1]  # sab columns except last
y_lstm = input_sequences[:, -1]   # sirf last column

# One-hot encode y
y_lstm = tf.keras.utils.to_categorical(y_lstm, num_classes=vocab_size)

print("X_lstm shape:", X_lstm.shape)
print("y_lstm shape:", y_lstm.shape)

X_lstm shape: (904, 13)
y_lstm shape: (904, 488)


In [67]:
model_lstm = Sequential([
    Embedding(vocab_size, 64, input_length=max_seq_len - 1),
    LSTM(128),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

model_lstm.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_lstm.summary()

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [68]:
history = model_lstm.fit(
    X_lstm, y_lstm,
    epochs=100,
    verbose=1
)

print("Training done!")

Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0265 - loss: 6.1457
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0299 - loss: 5.7844
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0343 - loss: 5.6373
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0376 - loss: 5.6135
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0409 - loss: 5.5688
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0343 - loss: 5.5292
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0431 - loss: 5.4567
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0420 - loss: 5.3688
Epoch 9/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0520 - loss: 5.2894
Epoch 10/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0487 - loss: 5.2037
Epoch 11/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0631 - loss: 5.0935
Epoch 12/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy:

In [69]:
import pickle

# LSTM model save
model_lstm.save(r"C:\Users\ASUS\Intelligent-Credit-Risk-System\text_generator_model.h5")

# Tokenizer save
with open(r"C:\Users\ASUS\Intelligent-Credit-Risk-System\tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# max_seq_len save karo — generate_explanation.py mein kaam aayega
with open(r"C:\Users\ASUS\Intelligent-Credit-Risk-System\max_seq_len.pkl", "wb") as f:
    pickle.dump(max_seq_len, f)

print("text_generator_model.h5 saved!")
print("tokenizer.pkl saved!")
print("max_seq_len.pkl saved!")

text_generator_model.h5 saved!
tokenizer.pkl saved!
max_seq_len.pkl saved!
